# 09c — Manifest v4 on Alberta: the re-registered design (mirror of the parent's `11c_manifest_v4_freeze`; study plan v0.20; AB spec v0.6 D-AB14/D-AB15; zero solves)

Supersedes manifest v3.1 (kept as the disclosed prior registration, never deleted). The parent's v4 design (M4.38): **five
discretionary blocks at 20% each** — core habitat (macrorefugia, 1/v WITH a velocity floor per realization), structural
connectivity (transboundary current, CONVEX I², t = 1, no target: the R2 semantic gate), climate corridors (Carroll
centrality, identity), carbon (m_soc θ-target + biomass), biodiversity (birds + mammals) — the representativeness block and
naturalness unchanged; **seven scenarios** (balanced; core-habitat-, structural-connectivity-, climate-corridors-,
biodiversity-, carbon-forward; naturalness push) × two refugia futures = **14 voting cells**. Constants unchanged.

**What is Alberta's own here (AB spec v0.6):** the stack is the parent's v4 stack masked to Alberta on the same grid
(D-AB1; the AB dust rule re-run against AB totals, M2.3) — `input_data/aligned_stack_ab_v4/`; the weights are re-derived on
the Alberta stack under constant intended influence (the block shares are the parent's); **S4 keeps Alberta's pre-registered
ladder target (m_soc t = 0.772, θ 2×; D3 / M5.7) — D-AB14**; the EFG targets are the D-AB12 window targets, unchanged; the
budget is level A (D-AB5 v2). Kernel `y2y-geo`, minutes.

**This notebook = the parent's pre-solve steps (1)–(4) on Alberta:** (1) velocity pre-checks on the AB extent and the two
re-shaped layers, proven cell-exact against the v3.1 Alberta layers (floored 1/v = the capped v3.1 layer; I² = the v3.1
layer squared, up to the dust rules); (2) the re-audit of floored refugia and squared transboundary current under the frozen
constants (feature cards; the semantic gate withholds transboundary's target); (3) five-block accounting at 20% and the seven
scenarios' (w, t) pairs → `spec/scenarios_ab_v4.json`; (4) **`spec/manifest_v4.csv`** frozen with its sha (write-once), the
14 design rows carrying the parent's v4 columns (`block_structure`, `value_shapes`, `macrorefugia_path`, `reference_cell`,
`unguarded_probes`) beside the Alberta ones (level, budget, `applied_band_g` = the D-AB13 fallback, provenance; commit
`config.py` first — the freeze asserts the pipeline modules clean vs HEAD).

Run order after this: **10** (anchors, twins, guarded MGA at 5% and 2% for all 14, the reference cell's unguarded probes) →
**11** (D-AB13 on the guarded 5% tier) → **11b** → **11c** → **12** → **13** → **14**. Ethan runs every notebook in VS Code.


In [1]:
# ---- bootstrap: the parent's v4 stack, the Alberta v3.1 stack, the five blocks --------------------------------------------
import hashlib, importlib, json, pathlib, shutil, subprocess, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile
import rasterio.shutil as rio_shutil
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)
assert config.Y2Y_VERSION == "v4", f"config.Y2Y_VERSION is {config.Y2Y_VERSION}: manifest v4 is built with the v4 switch active"
VP, V31 = config.ab_paths("v4"), config.ab_paths("v3.1")
assert VP.version == "v4" and config.EFG_SUBDIR == VP.efg_subdir == "iucn_efg_v3", (VP.version, config.EFG_SUBDIR)
HERE = ROOT / "analyses" / "alberta_prioritization"; SPEC, DATA = HERE / "spec", HERE / "data"
REC = VP.records; REC.mkdir(exist_ok=True, parents=True)
CARDS = HERE / "audit" / "feature_cards_ab_v4"; CARDS.mkdir(exist_ok=True, parents=True)
SRC, DST = V31.stack, VP.stack                                   # aligned_stack_ab/ (v1 + v3.1 record)  ->  aligned_stack_ab_v4/
assert DST == config.AB_STACK_DIR and (SRC / VP.efg_subdir).is_dir(), (DST, VP.efg_subdir)
Y4 = config.y2y_paths("v4"); PSTACK = Y4.stack                  # the parent's v4 stack (its 11c): the source of the two re-shaped layers
assert (PSTACK / "transboundary_connectivity.tif").exists() and (Y4.records / "stack_v4_layer_sha256.json").exists(), \
    "the parent's v4 stack / record is absent -- run analyses/y2y/11c_manifest_v4_freeze first"
PMETA = json.loads((Y4.records / "stack_v4_layer_sha256.json").read_text())
V_FLOOR, SPAN_YR = PMETA["velocity_floor"]["v_floor_km_per_yr"], PMETA["velocity_floor"]["span_yr"]      # 1 cell / 110 yr = 0.00909 km/yr
BLK = config.BLOCKS_FIVE; assert config.BLOCKS is BLK, "config.BLOCKS is not the five-block structure -- Y2Y_VERSION must be v4"
CONT = lc.continuous_features()                                  # the 8 continuous features in manifest order
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
def read(p):
    with rasterio.open(p) as s:
        return s.read(1, masked=True).astype("float64").filled(np.nan)
PU = lc.pu_mask(SRC); n_pu = int(PU.sum())
with rasterio.open(SRC / "mask_protected_areas.tif") as s:
    LOCKED = (s.read(1) == 1) & PU
DISC = PU & ~LOCKED
with rasterio.open(SRC / "cost_uniform.tif") as s:
    PROF = dict(width=s.width, height=s.height, transform=s.transform, crs=s.crs)
def write_cog(path, arr, nodata, dtype):                         # the 01 recipe: in-memory GTiff -> COG
    ov = "nearest" if np.dtype(dtype) == np.uint8 else "average"
    prof = dict(driver="GTiff", count=1, dtype=dtype, nodata=nodata, **PROF)
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with MemoryFile() as mem:
        with mem.open(**prof) as tmp:
            tmp.write(np.asarray(arr).astype(dtype), 1)
            rio_shutil.copy(tmp, path, driver="COG", compress="DEFLATE", overview_resampling=ov, BIGTIFF="IF_SAFER")
print(f"VERSION {VP.version} | Alberta stack {SRC.name} -> {DST.name} (re-shaped layers from the parent's {PSTACK.name}) | PU {n_pu:,} cells, allocatable {int(DISC.sum()):,} | blocks {list(BLK)} | floor {V_FLOOR:.5f} km/yr over {SPAN_YR:.0f} yr")


VERSION v4 | Alberta stack aligned_stack_ab -> aligned_stack_ab_v4 (re-shaped layers from the parent's aligned_stack_v4) | PU 85,133 cells, allocatable 57,161 | blocks ['core_habitat', 'structural_connectivity', 'climate_corridors', 'carbon', 'biodiversity'] | floor 0.00909 km/yr over 110 yr


In [2]:
# ---- (1) the Alberta v4 stack: the parent's v4 layers masked to Alberta on the SAME grid (D-AB1), the AB dust rule re-run (M2.3) ----
# The two re-shaped layers are read from the parent's v4 stack and masked, exactly as 01 built the v1 stack from the parent's
# registered stack -- then PROVEN against the v3.1 Alberta layers: floored 1/v == min(v3.1 layer, 1/floor) cell for cell (the floor
# is a per-cell cap); I^2 == (v3.1 layer)^2 wherever neither dust rule fired (the parent's rule ran against the Y2Y total, the AB
# rule runs against the AB total; cells where exactly one fired are counted and disclosed). Every other layer is copied byte-identical.
dust_log = {}
def mask_dust(name, a):
    out = np.where(PU, a, np.nan).astype("float32"); tot = float(np.nansum(out))
    dust = np.isfinite(out) & (out > 0) & (out < config.DUST_SHARE_MIN * tot)
    n, m = int(dust.sum()), float(np.nansum(out[dust])); out[dust] = 0.0
    share = m / tot if tot else 0.0
    assert share < 1e-4, f"{name}: dust rule dropped real mass ({share:.1e}) -- STOP and inspect"
    dust_log[name] = dict(cells=n, mass_share=share, ab_total=tot)
    return out
# (1a) refugia: velocity pre-checks on the AB extent, then the floored layer per realization
RAWP = {"585": config.ALIGNED_DIR / "climate_scenarios" / "585_2071_2100.tif", "245": config.ALIGNED_DIR / "climate_scenarios" / "245_2071_2100.tif"}
SRCP = {"585": SRC / "climate_type_macrorefugia.tif", "245": SRC / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
PARP = {"585": PSTACK / "climate_type_macrorefugia.tif", "245": PSTACK / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
OUTP = {"585": DST / "climate_type_macrorefugia.tif", "245": DST / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
NAME = {"585": "climate_type_macrorefugia", "245": "climate_realizations/macrorefugia_245_2071_2100"}
rows = []
for lv in ("585", "245"):
    v = read(RAWP[lv]); vp = v[PU]; x31 = read(SRCP[lv])[PU]; x4 = read(PARP[lv])
    assert np.isfinite(vp).all() and (vp > 0).all(), f"{lv}: raw velocity has non-finite or non-positive cells on the AB PU"
    n_at = {k: int((vp <= m * V_FLOOR).sum()) for k, m in (("le_floor", 1), ("le_2x_floor", 2), ("le_5x_floor", 5), ("le_10x_floor", 10))}
    out = mask_dust(NAME[lv], x4); op = out[PU]; ok = np.isfinite(op) & (op > 0)
    assert np.allclose(op[ok], (1.0 / np.maximum(vp, V_FLOOR))[ok], rtol=1e-6), f"{lv}: the masked parent v4 layer is not 1/max(v, floor) on the AB PU"
    capped = np.minimum(x31, 1.0 / V_FLOOR)
    n_diff31 = int((~np.isclose(op, capped, rtol=1e-6, equal_nan=True)).sum())
    assert n_diff31 == dust_log[NAME[lv]]["cells"], f"{lv}: the floored layer differs from the capped v3.1 Alberta layer beyond the dust rule ({n_diff31} cells)"
    changed = int((~np.isclose(op, x31, rtol=1e-9, equal_nan=True)).sum())
    rows.append(dict(realization=f"ssp{lv}_2071_2100", span_yr=SPAN_YR, v_floor_km_per_yr=round(V_FLOOR, 6), min_v_km_per_yr=float(vp.min()),
                     p01_v=float(np.percentile(vp, 1)), median_v=float(np.median(vp)), n_zero=int((vp == 0).sum()), **n_at,
                     max_inv_v_v31=float(np.nanmax(x31)), max_inv_v_floored=float(np.nanmax(op)), cells_changed_by_floor=changed,
                     floor_inert=bool(changed == 0), cells_differing_from_capped_v31=n_diff31))
    write_cog(OUTP[lv], out, np.nan, "float32")
PRE = pd.DataFrame(rows); PRE.to_csv(REC / "velocity_precheck.csv", index=False)
pd.set_option("display.width", 220); print("velocity pre-checks on the Alberta PU:"); print(PRE.to_string(index=False))
print("floor verdict on Alberta:", "INERT on both realizations -- the v4 layers equal the v3.1 Alberta layers cell for cell" if PRE.floor_inert.all()
      else f"ACTIVE -- {PRE.cells_changed_by_floor.tolist()} cells capped (585 / 245); reported, not a decision")
# (1b) structural connectivity: the parent's I^2 masked to Alberta, the AB dust rule re-applied, proven against the v3.1 layer squared
c31 = read(SRC / "transboundary_connectivity.tif")[PU]; c4 = read(PSTACK / "transboundary_connectivity.tif")
sq = mask_dust("transboundary_connectivity", c4); s4 = sq[PU]; s31 = np.where(np.isfinite(c31), c31 ** 2, np.nan)
both = (s31 > 0) & (s4 > 0); assert np.allclose(s31[both], s4[both], rtol=1e-5), "I^2 differs from the v3.1 Alberta layer squared where neither dust rule fired -- STOP"
one_zero = int(((s31 > 0) != (s4 > 0)).sum()); one_zero_mass = float(np.nansum(s31[(s31 > 0) != (s4 > 0)]) / np.nansum(s31))   # the parent's rule ran against the Y2Y total of a squared layer
r31, r4 = float(np.nanmax(c31) / np.nanmean(c31)), float(np.nanmax(s4) / np.nanmean(s4))
write_cog(DST / "transboundary_connectivity.tif", sq, np.nan, "float32")
print(f"\ntransboundary I^2 on Alberta: {one_zero:,} cells where exactly one dust rule fired (parent Y2Y-total vs AB-total; {one_zero_mass:.1e} of the squared mass; disclosed) | "
      f"AB dust zeroed {dust_log['transboundary_connectivity']['cells']:,} cells | max/mean on the AB PU {r31:.1f} (v3.1, linear) -> {r4:.1f} (squared)")
# (1c) every other layer byte-identical from the v3.1 Alberta stack; the curated block folder; the PA mask; the cost layer
CHANGED = {"climate_type_macrorefugia.tif", "transboundary_connectivity.tif"}
HASHES = {"changed": {}, "unchanged": {}, "realizations": {}, "efg": {}, "parent_v4_source": {}}
for p in sorted(SRC.glob("*.tif")):
    if p.name in CHANGED:
        continue
    q = DST / p.name
    if not q.exists() or sha(q) != sha(p):
        shutil.copy2(p, q)
    assert sha(q) == sha(p), p.name
    HASHES["unchanged"][p.stem] = sha(q)
for name in sorted(CHANGED):
    HASHES["changed"][pathlib.Path(name).stem] = dict(v4=sha(DST / name), v31=sha(SRC / name)); HASHES["parent_v4_source"][pathlib.Path(name).stem] = sha(PSTACK / name)
for p in sorted((SRC / "climate_realizations").glob("*.tif")):
    q = DST / "climate_realizations" / p.name
    if p.name == "macrorefugia_245_2071_2100.tif":
        HASHES["realizations"][p.stem] = dict(v4=sha(q), v31=sha(p)); HASHES["parent_v4_source"][p.stem] = sha(PARP["245"]); continue
    if not q.exists() or sha(q) != sha(p):
        q.parent.mkdir(exist_ok=True); shutil.copy2(p, q)
    HASHES["realizations"][p.stem] = sha(q)
efg_dst = DST / VP.efg_subdir; efg_dst.mkdir(exist_ok=True)
for p in lc.efg_paths(SRC):
    q = efg_dst / p.name
    if not q.exists() or sha(q) != sha(p):
        shutil.copy2(p, q)
    assert sha(q) == sha(p), p.name
    HASHES["efg"][p.stem] = sha(q)
assert len(HASHES["efg"]) == 13 and len(HASHES["unchanged"]) >= 9, (len(HASHES["efg"]), len(HASHES["unchanged"]))
META = dict(created_utc=datetime.now(timezone.utc).isoformat(), source_ab=str(SRC.relative_to(ROOT)), source_parent_v4=str(PSTACK.relative_to(ROOT)), stack=str(DST.relative_to(ROOT)),
            n_ab_pu=n_pu, dust_share_min=config.DUST_SHARE_MIN, dust=dust_log, velocity_floor=dict(span_yr=SPAN_YR, v_floor_km_per_yr=V_FLOOR),
            transboundary_sq_one_side_dust_cells=one_zero, transboundary_sq_one_side_dust_mass_share=one_zero_mass, **HASHES)
(REC / "stack_ab_v4_layer_sha256.json").write_text(json.dumps(META, indent=1)); (DST / "_build_meta.json").write_text(json.dumps(META, indent=1))
mpath = config.write_manifest(analysis="ab_y2y", handoff_dir=DST, manifest_path=DST / "manifest.json")
MJ = json.loads(pathlib.Path(mpath).read_text()); PJ = json.loads((PSTACK / "manifest.json").read_text())
assert MJ["grid"] == PJ["grid"], "the Alberta v4 grid differs from the parent's v4 grid -- STOP"
assert all(pathlib.Path(ROOT / l["path"]).exists() and str(l["path"]).startswith(str(DST.relative_to(ROOT))) for l in MJ["layers"]), "manifest points outside the Alberta v4 stack"
assert (lc.pu_mask(DST) == PU).all(), "the v4 stack changed the Alberta planning-unit mask"
roles = pd.Series([l["role"] for l in MJ["layers"]]).value_counts().to_dict()
print(f"\nAlberta v4 stack: {len(HASHES['unchanged'])} unchanged layers (sha-identical), 2 re-shaped (+ the 245 realization), {len(HASHES['efg'])} EFG features | manifest {pathlib.Path(mpath).relative_to(ROOT)}: {roles} | grid identical to the parent's v4 stack")


velocity pre-checks on the Alberta PU:
     realization  span_yr  v_floor_km_per_yr  min_v_km_per_yr    p01_v  median_v  n_zero  le_floor  le_2x_floor  le_5x_floor  le_10x_floor  max_inv_v_v31  max_inv_v_floored  cells_changed_by_floor  floor_inert  cells_differing_from_capped_v31
ssp585_2071_2100    110.0           0.009091         0.296814 0.606352  1.868776       0         0            0            0             0       3.369113           3.369113                       0         True                                0
ssp245_2071_2100    110.0           0.009091         0.130209 0.342911  1.194142       0         0            0            0             0       7.679988           7.679988                       0         True                                0
floor verdict on Alberta: INERT on both realizations -- the v4 layers equal the v3.1 Alberta layers cell for cell

transboundary I^2 on Alberta: 5,225 cells where exactly one dust rule fired (parent Y2Y-total vs AB-total; 2.1e-05 of

In [3]:
# ---- (2) the re-audit under the frozen constants on the Alberta v4 stack: floored refugia (both realizations) + squared transboundary current ----
# R1-R4 as at AB-0a (config.AUDIT; the audit convention = 30% of the extent, M2.6), on the v4 stack, beside the v3.1 Alberta
# classification. Expected (the parent's): refugia diffuse-linear, unchanged; transboundary I^2 crosses theta / a_min
# (concentrated-satiating, an R2 target) -- and the R2 SEMANTIC GATE, registered in the parent's protocol (M4.38), withholds the
# target: pinch points are PLACES (the claim is the set of cells), secured by the convex shape on the linear arm with t = 1.
GATE = {"climate_type_macrorefugia": ("weight (diffuse-linear)", "value = residence time within the horizon, linear; floor = numerics hygiene"),
        "transboundary_connectivity": ("weight, t = 1 -- R2 SEMANTIC GATE withholds the target", "pinch points are place-semantic: the claim IS the set of cells; a target secures an amount and un-pins them (parent PF-2: 100% -> 7%)")}
rows = []
for name in ("climate_type_macrorefugia", "transboundary_connectivity"):
    r = lc.classify(name, handoff_dir=DST); r31 = lc.classify(name, handoff_dir=SRC)
    r.update(realization="ssp585" if name.startswith("climate") else "-", shape={"climate_type_macrorefugia": "1/max(v, floor)", "transboundary_connectivity": "I^2"}[name],
             registered_lever=GATE[name][0], gate_reason=GATE[name][1], v31_cls=r31["cls"], v31_leverage=r31["leverage"], v31_theta_area=r31["theta_area"], v31_target=r31["target"])
    rows.append(r); lc.feature_card(name, CARDS, handoff_dir=DST)
x245 = read(OUTP["245"])[PU]; cls, lever, target, d = lc.classify_values(x245)
rows.append(dict(feature="climate_type_macrorefugia", orient="reciprocal", realization="ssp245", shape="1/max(v, floor)", cls=cls, lever=lever, target=target, **d,
                 registered_lever=GATE["climate_type_macrorefugia"][0], gate_reason=GATE["climate_type_macrorefugia"][1]))
AUD = pd.DataFrame(rows); AUD.to_csv(REC / "audit_cards_v4.csv", index=False)
print(AUD[["feature", "realization", "shape", "cls", "lever", "target", "leverage", "theta_area", "theta_target", "v31_cls", "v31_leverage", "registered_lever"]].to_string(index=False))
ref = AUD[AUD.feature == "climate_type_macrorefugia"]; tb = AUD[AUD.feature == "transboundary_connectivity"].iloc[0]
print(f"\nrefugia on Alberta: {' / '.join(ref.cls)} (expected diffuse-linear, unchanged) | transboundary I^2: {tb.cls}, R2 target {tb.target} -> registered lever: {tb.registered_lever}")
print(f"cards -> {CARDS.relative_to(ROOT)}/ ; table -> {REC.relative_to(ROOT)}/audit_cards_v4.csv")


                   feature realization           shape                    cls  lever  target  leverage  theta_area  theta_target        v31_cls  v31_leverage                                       registered_lever
 climate_type_macrorefugia      ssp585 1/max(v, floor)         diffuse-linear weight   1.000     0.309      0.0000         0.000 diffuse-linear         0.309                                weight (diffuse-linear)
transboundary_connectivity           -             I^2 concentrated-satiating target   0.267     0.791      0.0265         0.267 diffuse-linear         0.523 weight, t = 1 -- R2 SEMANTIC GATE withholds the target
 climate_type_macrorefugia      ssp245 1/max(v, floor)         diffuse-linear weight   1.000     0.341      0.0003         0.002            NaN           NaN                                weight (diffuse-linear)

refugia on Alberta: diffuse-linear / diffuse-linear (expected diffuse-linear, unchanged) | transboundary I^2: concentrated-satiating, R2 target 0.2

In [4]:
# ---- (3) five-block accounting at 20% each on the Alberta stack; the seven scenarios' (w, t) pairs under constant intended influence ----
# Doubling rule (the parent's): the forward block takes 40%, the other four 15% each. Carbon's inner split = the measured mass
# split on the Alberta extent (scenarios_ab_v1: frac_soc); biodiversity equal (D-AB9: both AOH members stay). Targets: S0-S3 /
# S2c / S5 m_soc t = 0.322 (D2 mirror; pre-satisfied on Alberta); **S4 = Alberta's pre-registered ladder target 0.772 (theta 2x;
# D3 / M5.7) with the carbon block doubled -- D-AB14: the ladder was decided against the MEASURED Alberta co-capture floor and
# is not re-derived from the parent's theta 3x**; S5 = S0 with naturalness pushed x10. Weights are derived per climate cell from
# the v4 stack (the 245 cells swap the floored 245 refugia layer in); realized == intended is asserted inside scenario_weights.
SC1 = json.loads((SPEC / "scenarios_ab_v1.json").read_text())
WITHIN = {"carbon": SC1["S0_balanced"]["within_block"]["carbon"]}
T0, T4 = dict(SC1["S0_balanced"]["targets"]), dict(SC1["S4_carbon"]["targets"])
LADDER = SC1["_meta"]["s4_ladder"]; assert abs(T4["irrecoverable_carbon_m_soc"] - LADDER["chosen_target"]) < 1e-9
REGIME4 = f"theta{LADDER['chosen_theta']:g}_places"            # theta2_places on Alberta (the parent: theta3_places)
def shares(forward=None):
    return {b: (0.4 if b == forward else (0.15 if forward else 0.2)) for b in BLK}
SCEN = {"s0": dict(name="S0_balanced", label="Balanced", shares=shares(), targets=T0, extra={}, regime="theta5_amount"),
        "s1": dict(name="S1_core_habitat", label="Core-habitat-forward", shares=shares("core_habitat"), targets=T0, extra={}, regime="theta5_amount"),
        "s2": dict(name="S2_structural_connectivity", label="Structural-connectivity-forward", shares=shares("structural_connectivity"), targets=T0, extra={}, regime="theta5_amount"),
        "s2c": dict(name="S2c_climate_corridors", label="Climate-corridors-forward", shares=shares("climate_corridors"), targets=T0, extra={}, regime="theta5_amount"),
        "s3": dict(name="S3_biodiversity", label="Biodiversity-forward", shares=shares("biodiversity"), targets=T0, extra={}, regime="theta5_amount"),
        "s4": dict(name="S4_carbon", label="Carbon-forward", shares=shares("carbon"), targets=T4, extra={}, regime=REGIME4),
        "s5": dict(name="S5_naturalness", label="Naturalness push (S0 + gHM x10)", shares=shares(), targets=T0, extra={"human_modification": 10.0}, regime="theta5_amount")}
LP = {"ssp585_2071_2100": None, "ssp245_2071_2100": {"climate_type_macrorefugia": OUTP["245"]}}
W = {}
for sid, s in SCEN.items():
    for climate, lp in LP.items():
        d = lc.scenario_weights(s["shares"], within_block=WITHIN, targets=s["targets"], blocks=BLK, layer_paths=lp, handoff_dir=DST)
        w = {r.feature: round(float(r.w), 6) for r in d.itertuples()}; w.update(s["extra"])
        W[(sid, climate)] = dict(weights=w, intended={r.feature: round(float(r.intended_share), 6) for r in d.itertuples()},
                                 swing={r.feature: float(r.per_unit_swing) for r in d.itertuples()})
ORDER = [f for f in CONT if f != "human_modification"] + ["human_modification"]
tab = pd.DataFrame({f"{sid}@{c.split('_')[0][-3:]}": W[(sid, c)]["weights"] for sid in SCEN for c in LP}).reindex(ORDER).fillna(1.0)   # naturalness outside at 1
print("weights per formulation (Alberta v4 stack; constant intended influence; naturalness outside at 1, x10 under S5):"); print(tab.to_string(float_format=lambda v: f"{v:.4f}"))
V31W = pd.read_csv(V31.manifest).set_index("formulation_id")
w31 = json.loads(V31W.loc["s0_ssp585_theta5"].weight_vector)
print("\nS0@585 weights, v3.1 (four blocks at 25%) -> v4 (five at 20%): " + ", ".join(f"{k.split('_')[-1]} {w31.get(k, 1.0):.3f} -> {W[('s0', 'ssp585_2071_2100')]['weights'].get(k, 1.0):.3f}" for k in ORDER))
payload = dict(_meta=dict(derived_utc=datetime.now(timezone.utc).isoformat(), spec_version="AB v0.6 (mirror of study plan v0.20, manifest v4)", extent_id=SC1["_meta"]["extent_id"],
                          n_pu=n_pu, budget_pct_audit=SC1["_meta"]["budget_pct_audit"], budget_pct_solve_level_A=SC1["_meta"]["budget_pct_solve_level_A"],
                          blocks=BLK, doubling_rule="forward block 0.40, the other four 0.15; balanced 0.20 x 5", within_block=WITHIN, split_rule=SC1["_meta"]["split_rule"],
                          s4_ladder=LADDER, s4_decision="D-AB14: S4 keeps Alberta's ladder target (theta 2x, t 0.772) under the doubled carbon block",
                          biodiversity_block=SC1["_meta"]["biodiversity_block"],
                          shapes={"climate_type_macrorefugia": f"1/max(v, {V_FLOOR:.6f} km/yr)", "transboundary_connectivity": "I^2 (t = 1, R2 semantic gate)", "others": "identity"},
                          stack=str(DST.relative_to(ROOT)), n_efg=len(HASHES["efg"]),
                          normalization="derived weights rescaled to mean 1 over the blocked features; outside features at their baselines (gHM w=1, EFG 1/n)"),
               **{s["name"]: dict(scenario_id=sid, label=s["label"], block_shares=s["shares"], within_block=WITHIN, targets=s["targets"], extra=s["extra"], carbon_regime=s["regime"],
                                   weights={c: W[(sid, c)]["weights"] for c in LP}, intended_shares=W[(sid, "ssp585_2071_2100")]["intended"]) for sid, s in SCEN.items()})
(SPEC / "scenarios_ab_v4.json").write_text(json.dumps(payload, indent=2))
print(f"\nwrote {(SPEC / 'scenarios_ab_v4.json').relative_to(ROOT)} ({len(SCEN)} scenarios x {len(LP)} climate cells; S4 t = {T4['irrecoverable_carbon_m_soc']} at theta {LADDER['chosen_theta']:g}x, D-AB14)")


weights per formulation (Alberta v4 stack; constant intended influence; naturalness outside at 1, x10 under S5):
                              s0@585  s0@245  s1@585  s1@245  s2@585  s2@245  s2c@585  s2c@245  s3@585  s3@245  s4@585  s4@245  s5@585  s5@245
transboundary_connectivity    0.3260  0.3297  0.2719  0.2788  0.8067  0.8151   0.2022   0.2036  0.1914  0.1927  0.3034  0.3066  0.3260  0.3297
climate_corridors             2.5722  2.6013  2.1454  2.2002  2.3869  2.4120   4.2540   4.2837  1.5106  1.5206  2.3941  2.4193  2.5722  2.6013
climate_type_macrorefugia     0.8355  0.7657  1.8583  1.7270  0.7753  0.7100   0.5182   0.4728  0.4907  0.4476  0.7776  0.7121  0.8355  0.7657
irrecoverable_carbon_biomass  0.1384  0.1399  0.1154  0.1184  0.1284  0.1297   0.0858   0.0864  0.0813  0.0818  0.3434  0.3470  0.1384  0.1399
irrecoverable_carbon_m_soc    0.1764  0.1784  0.1472  0.1509  0.1637  0.1654   0.1094   0.1102  0.1036  0.1043  0.4343  0.4389  0.1764  0.1784
aoh_richness_mammals         

In [5]:
# ---- (4) manifest v4 on Alberta: 14 design formulations, the parent's v4 columns beside the Alberta ones, frozen with its sha (write-once) ----
MODULES = ["config.py", "leverage_core.py", "ensemble_core.py", "prioritizr_core.R", "mga_core.R"]
git = lambda *a: subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True)
GIT_SHA = git("rev-parse", "HEAD").stdout.strip()
dirty = [m for m in MODULES if git("diff", "--quiet", "HEAD", "--", m).returncode != 0]
assert not dirty, f"pipeline module(s) modified vs HEAD -- commit before freezing: {dirty}"
MOD_SHA = {m: hashlib.sha256((ROOT / m).read_bytes()).hexdigest() for m in MODULES}
sha_v31 = sha(V31.manifest); assert sha_v31 == V31.freeze.read_text().split()[0], "manifest v3.1 no longer matches its freeze hash -- stop"
M31 = pd.read_csv(V31.manifest).set_index("formulation_id"); assert len(M31) == 12 and (M31.role == "design").all()
T31 = json.loads((V31.records / "efg_targets.json").read_text())["targets"]                # the 13 D-AB12 window targets, unchanged under v4
T31 = {k: round(float(v), 4) for k, v in T31.items()}
assert len(T31) == 13 and all({k: round(float(v), 4) for k, v in json.loads(r.target_vector).items() if k not in CONT} == T31 for _, r in M31.iterrows()), "the v3.1 rows do not all carry the D-AB12 targets"
EFG_SHA = hashlib.sha256("\n".join(f"{k}:{v}" for k, v in sorted(HASHES["efg"].items())).encode()).hexdigest()
assert EFG_SHA == M31.iloc[0].efg_block_sha256, "the Alberta v4 stack's EFG block differs from v3.1's"
LAYER_SHA = {f: sha(DST / f"{f}.tif") for f in CONT}
SHAPES = payload["_meta"]["shapes"]; REF = "s0_ssp585_theta5"; PROBES = [0.02, 0.05, 0.10]
APPLIED_RULE = ("D-AB13: guarded g = 5% (mirror) unless the 5% guarded frequent tier is flat (< 100 km2), then g = 2% (D-AB10); decided in 11. "
                "D-AB15 (v4): the guarded band is the estimand (parent M4.38) -- guarded MGA at 5% AND at 2% for every formulation; the unguarded band on the reference cell only (its probes)")
now = datetime.now(timezone.utc).isoformat(); rows = []
for i, (climate, lp) in enumerate(LP.items()):
    for j, (sid, s) in enumerate(SCEN.items()):
        fid = f"{sid}_{climate.split('_')[0]}_{s['regime'].split('_')[0]}"
        tsid = sid if sid != "s2c" else "s2"                                                 # s2c is new: its Alberta columns come from the s2 row of the same climate
        tfid = next(f for f in M31.index if f.startswith(f"{tsid}_{climate.split('_')[0]}_"))
        row = M31.loc[tfid].to_dict(); row["formulation_id"] = fid
        hashes = {**{k: v for k, v in LAYER_SHA.items() if k != "climate_type_macrorefugia"},
                  "climate_type_macrorefugia": HASHES["changed"]["climate_type_macrorefugia"]["v4"] if lp is None else HASHES["realizations"]["macrorefugia_245_2071_2100"]["v4"], **HASHES["efg"]}
        row.update(scenario_id=sid, scenario_name=s["name"], climate_level=climate, carbon_regime=s["regime"], mirror_spec_version="v0.6", role="design",
                   weight_vector=json.dumps(W[(sid, climate)]["weights"]), target_vector=json.dumps({**s["targets"], **T31}),
                   influence_profile_intended=json.dumps(W[(sid, climate)]["intended"]), k_requested=50, band_gap_g=0.05, floor_g=0.05, applied_band_g=0.02, applied_band_rule=APPLIED_RULE,
                   opt_gap=1e-4, numeric_focus=2, solver="gurobi", anchor_ref="", twin_ref="", mga_ref="", input_layer_hashes=json.dumps(hashes),
                   block_structure=json.dumps(BLK), value_shapes=json.dumps(SHAPES),
                   macrorefugia_path=str((OUTP["585"] if lp is None else OUTP["245"]).relative_to(ROOT)),
                   reference_cell=bool(fid == REF), unguarded_probes=json.dumps(PROBES if fid == REF else []), e12_seed=20260921 + i * len(SCEN) + j,
                   manifest_version=4, efg_block_version="v3", efg_block_sha256=EFG_SHA, efg_features=json.dumps(sorted(HASHES["efg"])),
                   supersedes=f"v3.1 {sha_v31[:16]}",
                   trigger="AB spec v0.6 (parent study plan v0.20 / M4.38, E20 + PF-4): five blocks at 20%, transboundary I^2 (no target), refugia velocity floor, S2c; S4 at Alberta's ladder (D-AB14)",
                   pipeline_git_sha=GIT_SHA, pipeline_module_sha256=json.dumps(MOD_SHA), created_utc=now, frozen=True)
        rows.append(row)
M4 = pd.DataFrame(rows); assert M4.formulation_id.is_unique and len(M4) == 14 and M4.reference_cell.sum() == 1
for fn in ("efg_targets.json", "efg_window_footprints.csv", "efg_block_card_v3.csv", "efg_curation_v3.csv"):     # the EFG records the readers expect beside the version's records (unchanged from v3.1)
    if not (REC / fn).exists():
        shutil.copy2(V31.records / fn, REC / fn)
if VP.manifest.exists() and VP.freeze.exists() and sha(VP.manifest) == VP.freeze.read_text().split()[0]:
    d4 = VP.freeze.read_text().split()[0]
    print(f"manifest v4 already frozen ({d4[:16]}...) -- kept byte-identical (supersede, never delete); delete both files to re-freeze")
else:
    M4.to_csv(VP.manifest, index=False); d4 = sha(VP.manifest); VP.freeze.write_text(f"{d4}  {VP.manifest.name}\n")
    print(f"FROZEN: {VP.manifest.relative_to(ROOT)} (14 design formulations at level {M4.budget_level.iloc[0]}; blocks {list(BLK)}; EFG block sha {EFG_SHA[:16]}...) | sha256 {d4[:16]}...")
print(M4[["formulation_id", "scenario_id", "climate_level", "carbon_regime", "reference_cell", "unguarded_probes", "applied_band_g", "macrorefugia_path"]].to_string(index=False))
print("\ncommit spec/manifest_v4.csv + spec/manifest_v4.sha256 + spec/v4/ + spec/scenarios_ab_v4.json -- that commit is the pre-registration of the v4 re-solve; then 10_ab4_ensemble (R, VERSION v4)")


FROZEN: analyses/alberta_prioritization/spec/manifest_v4.csv (14 design formulations at level A; blocks ['core_habitat', 'structural_connectivity', 'climate_corridors', 'carbon', 'biodiversity']; EFG block sha c8ee930b64e00bca...) | sha256 0c2a07574701e8b4...
   formulation_id scenario_id    climate_level carbon_regime  reference_cell  unguarded_probes  applied_band_g                                                                  macrorefugia_path
 s0_ssp585_theta5          s0 ssp585_2071_2100 theta5_amount            True [0.02, 0.05, 0.1]            0.02                       input_data/aligned_stack_ab_v4/climate_type_macrorefugia.tif
 s1_ssp585_theta5          s1 ssp585_2071_2100 theta5_amount           False                []            0.02                       input_data/aligned_stack_ab_v4/climate_type_macrorefugia.tif
 s2_ssp585_theta5          s2 ssp585_2071_2100 theta5_amount           False                []            0.02                       input_data/aligned_stack_

## → next

Commit the freeze, then `10_ab4_ensemble` (R; `VERSION <- "v4"`: anchors + twins for all 14, guarded MGA at 5% and 2% for all 14, the
reference cell's unguarded probes at 2 / 5 / 10%) → `11_ab4_analysis` (D-AB13 on the guarded 5% tier; the guarded band is the estimand)
→ `11b_ab_e19_solves` (R: T2 ×14, E17-T3 six arms) → `11c_ab_e19_analysis` → `12_tiers_and_clusters` → `13_figures` → `14_director_outputs`.
Numeric order = run order; Ethan runs each in VS Code. Log the pre-checks, the re-audit, the weights and the freeze in `spec/results_log.md` (R9.1).
